## Define Functions

In [ ]:
import plotly.io as pio

#pio.renderers.default = "notebook"  # embeds interactive figures inline  [oai_citation:0‡plotly.com](https://plotly.com/python/renderers/?utm_source=chatgpt.com)

import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import plotly.express as px
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.stats.multitest import multipletests
import scanpy as sc
import squidpy as sq
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.io as pio
from anndata import AnnData
import anndata as ad


def plot_adata(adata_to_plot, cluster_key, plot_type="umap"):
    if plot_type == "spatial":

        coords = adata_to_plot.obsm['spatial'].copy()
        df = pd.DataFrame(coords, columns=['x', 'y'], index=adata_to_plot.obs_names)

        df[cluster_key] = adata_to_plot.obs[cluster_key].astype(str)

        tab20 = [mpl.colors.rgb2hex(c) for c in plt.get_cmap('tab20').colors]

        fig = px.scatter(
            df,
            x='x',
            y='y',
            color=cluster_key,
            title='Spatial scatter — manual cell types',
            #color_discrete_sequence=tab20,
            hover_name=df.index,
            width=1600,
            height=700
        )

        fig.update_traces(marker=dict(size=2, opacity=0.8))
        fig.update_yaxes(autorange='reversed')
        fig.update_layout(
            legend_title_text=cluster_key,
            legend=dict(
                itemsizing='constant',
                traceorder='normal',
                bgcolor='rgba(255,255,255,0.5)',
                x=1.02, y=1
            ),
            margin=dict(l=20, r=200, t=50, b=20)
        )

        fig.show()



    elif plot_type == "umap":

        df = pd.DataFrame(
            adata_to_plot.obsm['X_umap'],
            columns=['UMAP1', 'UMAP2'],
            index=adata_to_plot.obs_names
        ).copy()

        df[cluster_key] = adata_to_plot.obs[cluster_key].astype(str)

        tab20 = [mpl.colors.rgb2hex(c) for c in plt.get_cmap('tab20').colors]

        fig = px.scatter(
            df,
            x='UMAP1',
            y='UMAP2',
            color=cluster_key,
            title='UMAP embedding — Leiden clusters',
            color_discrete_sequence=tab20,
            hover_name=df.index,
            width=1400,
            height=1200
        )

        fig.update_traces(marker=dict(size=3, opacity=0.8))
        fig.update_layout(
            legend_title_text='Leiden cluster',
            legend=dict(
                itemsizing='constant',
                traceorder='normal',
                bgcolor='rgba(255,255,255,0.5)',
                x=1.02, y=1
            ),
            margin=dict(l=20, r=200, t=50, b=20)
        )

        fig.show()



import scanpy as sc
import gseapy as gp

def go_fgsea(adata, ref, comp, gene_set="MSigDB_Hallmark_2020", io_key = "assigned_celltype_L0", show_plot = True):
    Reference, Delta = adata[adata.obs[io_key] == ref], \
                       adata[adata.obs[io_key] == comp]
    
    adata_concat = sc.concat(
        [Reference, Delta],
        axis=0,
        join="inner",                # keep only shared variables
        label="subset",              # name of the new obs‐column
        keys=[ref, comp],
        merge="same"                 # assume var‐ and obsm‐entries are identical
    )

        # 1) Run differential expression: PartialTumor vs TumorEnriched
    sc.tl.rank_genes_groups(
    adata_concat,
    groupby='subset',
    groups=[comp],
    reference=ref,
    method='t-test'          # or 't-test', 'logreg', etc.
    )

    # 2) Extract a pre-ranked list of genes (log₂-fold-changes)
    de_df = sc.get.rank_genes_groups_df(
    adata_concat,
    group=comp,
    key='rank_genes_groups'
    )
    rnk = de_df.set_index('names')['logfoldchanges']

    pre_res = gp.prerank(
    rnk=rnk,
    gene_sets=gene_set,     # or path to your GMT file
    processes=4,
    permutation_num=1000,
    outdir=None)


    if show_plot:
        sc.pl.umap(
        adata_concat,
        color="subset",
        palette=["red","blue"],
        size=20,
        title=f"UMAP Comparisons of {ref} as reference vs {comp} as comparison",
        alpha = 0.4)

        terms = pre_res.res2d.Term
        axs = pre_res.plot(terms[:5], show_ranking=False, legend_kws={'loc': (1.05, 0)}, )

    return pre_res.res2d[pre_res.res2d["FDR q-val"] < 0.2] 


def de_volcano(
    adata,
    groupby: str,
    ref: str,
    comp: str,
    method: str = "t-test",
    pval_adj_method: str = "fdr_bh",
    lfc_thresh: float = 1.0,
    pval_thresh: float = 0.05,
    top_n: int = 10,
    show: bool = True
) -> pd.DataFrame:
    """
    Run group-level DE, display a volcano plot with gene labels for the top N hits.

    Returns a DataFrame with DE results including names, logfoldchanges, pvals, pvals_adj.
    """
    # 1) Subset and run DE
    ad = sc.concat(
        [adata[adata.obs[groupby] == ref],
         adata[adata.obs[groupby] == comp]],
        join="inner", label="subset", keys=[ref, comp], merge="same"
    )
    sc.tl.rank_genes_groups(
        ad, groupby="subset", groups=[comp], reference=ref, method=method
    )

    # 2) Build DE results table
    r = sc.get.rank_genes_groups_df(ad, group=comp)
    de_df = r[['names','logfoldchanges','pvals']].copy()
    de_df['pvals_adj'] = multipletests(de_df['pvals'], method=pval_adj_method)[1]

    if show:
        x = de_df['logfoldchanges']
        y = -np.log10(de_df['pvals_adj'] + 1e-300)

        plt.figure(figsize=(6,6))
        # all points
        plt.scatter(x, y, c='lightgrey', s=10, alpha=0.6)
        # significant
        sig = (np.abs(x) >= lfc_thresh) & (de_df['pvals_adj'] <= pval_thresh)
        plt.scatter(
            x[sig], y[sig],
            c=np.where(x[sig]>0, 'red','blue'),
            s=20, alpha=0.8
        )
        # threshold lines
        plt.axvline(lfc_thresh,  color='grey', linestyle='--', linewidth=1)
        plt.axvline(-lfc_thresh, color='grey', linestyle='--', linewidth=1)
        plt.axhline(-np.log10(pval_thresh), color='grey', linestyle='--', linewidth=1)

        # annotate top N by adjusted p-value
        top_hits = de_df.nsmallest(top_n, 'pvals_adj')
        for _, row in top_hits.iterrows():
            xi = row['logfoldchanges']
            yi = -np.log10(row['pvals_adj'] + 1e-300)
            plt.text(xi, yi, row['names'], fontsize=8,
                     ha='right' if xi<0 else 'left')

        plt.xlabel('log₂ fold-change')
        plt.ylabel('-log₁₀ adjusted p-value')
        plt.tight_layout()
        plt.show()

    return de_df

# Ensure inline rendering in Jupyter
pio.renderers.default = "notebook"

def plot_adata(adata_to_plot, cluster_key, plot_type="umap", keyword=None):
    """
    Plot AnnData embeddings (UMAP or spatial) in Jupyter Notebook, with
    the X axis extended to 3× its data span.
    """
    # Prepare DataFrame
    if plot_type == "spatial":
        coords = adata_to_plot.obsm['spatial']
        df = pd.DataFrame(coords, columns=['x','y'], index=adata_to_plot.obs_names)
        xcol, ycol = 'x', 'y'
        title = 'Spatial'
    else:
        df = pd.DataFrame(
            adata_to_plot.obsm['X_umap'],
            columns=['UMAP1','UMAP2'],
            index=adata_to_plot.obs_names
        )
        xcol, ycol = 'UMAP1', 'UMAP2'
        title = 'UMAP'

    df[cluster_key] = adata_to_plot.obs[cluster_key].astype(str)

    if keyword:
        df['plot_label'] = df[cluster_key].apply(lambda x: x if keyword in x else 'Other')
        color_key = 'plot_label'
        legend_title = f"Highlighted: {keyword}"
    else:
        color_key, legend_title = cluster_key, cluster_key

    # Discrete tab20 colors
    tab20 = [mpl.colors.rgb2hex(c) for c in plt.get_cmap('tab20').colors]

    # Build scatter
    fig = px.scatter(
        df,
        x=xcol,
        y=ycol,
        color=color_key,
        hover_name=df.index,
        title=title,
        color_discrete_sequence=tab20,
        width=1400,
        height=600
    )
    if plot_type == "spatial":
        fig.update_yaxes(autorange='reversed')

    fig.update_traces(marker=dict(size=6, opacity=0.8))
    fig.update_layout(
        legend_title_text=legend_title,
        legend=dict(bgcolor='rgba(255,255,255,0.5)', x=1.02, y=1),
        margin=dict(l=20, r=200, t=50, b=20),
        plot_bgcolor='white', paper_bgcolor='white'
    )

    # === New: extend X-axis to 3× its data span ===
    x_min, x_max = df[xcol].min(), df[xcol].max()
    span  = x_max - x_min
    center= (x_min + x_max) / 2
    half_new = 1.5 * span
    fig.update_xaxes(range=[center - half_new, center + half_new])

    # Force the notebook renderer
    fig.show(renderer="notebook")


def filter_adata_by_region(
    adata: AnnData,
    x_bounds: tuple[float, float],
    y_bounds: tuple[float, float],
    coord_key: str = "spatial"
) -> AnnData:
    """
    Subset AnnData to cells whose 2D coordinates lie within a rectangle.

    Parameters
    ----------
    adata
        Input AnnData with embedding or spatial coords in .obsm
    x_bounds
        (x_min, x_max) inclusive bounds on the first coordinate
    y_bounds
        (y_min, y_max) inclusive bounds on the second coordinate
    coord_key
        Key in adata.obsm containing an (n_obs, 2) array of coordinates
        e.g. "spatial" or "X_umap"

    Returns
    -------
    AnnData
        A new AnnData with only the cells inside the given region.
    """
    # Extract coords
    coords = adata.obsm.get(coord_key)
    if coords is None:
        raise KeyError(f"'{coord_key}' not found in adata.obsm")
    if coords.shape[1] < 2:
        raise ValueError(f"adata.obsm['{coord_key}'] must have at least 2 columns")

    x_min, x_max = x_bounds
    y_min, y_max = y_bounds

    # Build mask
    xs = coords[:, 0]
    ys = coords[:, 1]
    mask = (xs >= x_min) & (xs <= x_max) & (ys >= y_min) & (ys <= y_max)

    # Subset and return
    return adata[mask].copy()

def run_ligrec(adata):
    # all your parameters here
    return sq.gr.ligrec(
        adata,
        n_perms=200,
        cluster_key="assigned_celltype_L0",
        copy=True,
        use_raw=False,
        transmitter_params={"categories": "ligand"},
        receiver_params={"categories": "receptor"}
    )

def plot_ligrec(res,source,target):

    df_plot = res["means"][source][target].rename("value").reset_index()

    fig = px.scatter(
        df_plot,
        x="target",
        y="source",
        size="value",
        color="value",
        color_continuous_scale="Blues",
        size_max=20,
        hover_data=["value"],
        labels={"value": "Score"},
        width=1600,
        height=1000
    )
    fig.update_layout(
        xaxis_tickangle=45,
        yaxis=dict(tickfont=dict(size=8)),
        plot_bgcolor="white",
        margin=dict(l=200, r=20, t=50, b=200)
    )
    fig.show()





In [ ]:
# import pandas as pd
# import plotly.express as px
# import matplotlib.pyplot as plt
# import matplotlib as mpl


# def plot_adata(adata_to_plot, cluster_key, plot_type="umap"):
#     if plot_type == "spatial":

#         coords = adata_to_plot.obsm['spatial'].copy()
#         df = pd.DataFrame(coords, columns=['x', 'y'], index=adata_to_plot.obs_names)

#         df[cluster_key] = adata_to_plot.obs[cluster_key].astype(str)

#         tab20 = [mpl.colors.rgb2hex(c) for c in plt.get_cmap('tab20').colors]

#         fig = px.scatter(
#             df,
#             x='x',
#             y='y',
#             color=cluster_key,
#             title='Spatial scatter — manual cell types',
#             #color_discrete_sequence=tab20,
#             hover_name=df.index,
#             width=1600,
#             height=700
#         )

#         fig.update_traces(marker=dict(size=2, opacity=0.8))
#         fig.update_yaxes(autorange='reversed')
#         fig.update_layout(
#             legend_title_text=cluster_key,
#             legend=dict(
#                 itemsizing='constant',
#                 traceorder='normal',
#                 bgcolor='rgba(255,255,255,0.5)',
#                 x=1.02, y=1
#             ),
#             margin=dict(l=20, r=200, t=50, b=20)
#         )

#         fig.show()



#     elif plot_type == "umap":

#         df = pd.DataFrame(
#             adata_to_plot.obsm['X_umap'],
#             columns=['UMAP1', 'UMAP2'],
#             index=adata_to_plot.obs_names
#         ).copy()

#         df[cluster_key] = adata_to_plot.obs[cluster_key].astype(str)

#         tab20 = [mpl.colors.rgb2hex(c) for c in plt.get_cmap('tab20').colors]

#         fig = px.scatter(
#             df,
#             x='UMAP1',
#             y='UMAP2',
#             color=cluster_key,
#             title='UMAP embedding — Leiden clusters',
#             color_discrete_sequence=tab20,
#             hover_name=df.index,
#             width=1400,
#             height=1200
#         )

#         fig.update_traces(marker=dict(size=3, opacity=0.8))
#         fig.update_layout(
#             legend_title_text='Leiden cluster',
#             legend=dict(
#                 itemsizing='constant',
#                 traceorder='normal',
#                 bgcolor='rgba(255,255,255,0.5)',
#                 x=1.02, y=1
#             ),
#             margin=dict(l=20, r=200, t=50, b=20)
#         )

#         fig.show()

## Load Data

In [ ]:
import scanpy as sc
import squidpy as sq

Tissue = "Region2"

adata = sc.read_h5ad(f"/Volumes/ProstateCancerEvoMain/dbs/Completed/AllRegions/{Tissue}.raw.annotated.V2.h5ad")
adatayoutu

tile_df = pd.read_csv("Barcode_TileClone_Pair.V2.csv")
tile_df


In [ ]:
# 1) collapse duplicates by taking the first Clone/Tile per cell_id
unique_tiles = (
    tile_df
      .groupby("cell_id", as_index=False)
      .first()[["cell_id","Clone","Tile"]]
)

# 2) build maps from that unique table
clone_map = unique_tiles.set_index("cell_id")["Clone"]
tile_map  = unique_tiles.set_index("cell_id")["Tile"]

# 3) map into adata.obs without touching the index
adata.obs["Clone"] = adata.obs["cell_id"].map(clone_map)
adata.obs["Tile"]  = adata.obs["cell_id"].map(tile_map)

# sanity check
adata.obs[["cell_id","Clone","Tile"]]

In [ ]:
plot_adata(adata, "Clone",plot_type="spatial")

## Global Spatial Relationship Analysis

In [ ]:
src = "L0_leiden_assigned_celltype"
dst = "L0_leiden_assigned_celltype_sq"

adata.obs[dst] = (
    adata.obs[src]
    .astype(str)
    .where(
        adata.obs[src].str.contains("tumor", na=False),
        adata.obs[src].astype(str).str.replace(r'_[^_]+$', "", regex=True)
    )
    .astype("category")
)

In [ ]:
sq.gr.spatial_neighbors(adata, coord_type="generic", delaunay=True)

sq.gr.centrality_scores(adata, cluster_key="L0_leiden_assigned_celltype_sq")

sq.pl.centrality_scores(adata, palette= "tab20" ,cluster_key="L0_leiden_assigned_celltype_sq",figsize=(50,10))

In [ ]:
#Subsampling For Further Analysis

adata_subsample = sc.pp.subsample(adata, fraction=0.5, copy=True)
adata_subsample.obs["L0_leiden_assigned_celltype_sq"]

In [ ]:
sq.gr.co_occurrence(
    adata_subsample,
    cluster_key="L0_leiden_assigned_celltype_sq",
)

sq.pl.co_occurrence(
    adata_subsample,
    cluster_key="L0_leiden_assigned_celltype_sq",
    clusters=["tumor_cell_markers_1","tumor_cell_markers_3","tumor_cell_markers_11"],
    figsize=(20, 10),
)

In [ ]:
sq.pl.spatial_scatter(
    adata_subsample,
    color="L0_leiden_assigned_celltype_sq",
    shape=None,
    size=4,
)

In [ ]:
sq.gr.nhood_enrichment(adata, cluster_key="L0_leiden_assigned_celltype_sq")

sq.pl.nhood_enrichment(
    adata,
    cluster_key="L0_leiden_assigned_celltype_sq",
    title="Neighborhood Enrichment (Selected Groups)",
    figsize=(8, 8),
    palette="tab20",
    cmap="viridis"
)

In [ ]:
mode = "L"
sq.gr.ripley(adata, cluster_key="L0_leiden_assigned_celltype_sq", mode=mode)
sq.pl.ripley(adata, cluster_key="L0_leiden_assigned_celltype_sq", mode=mode)

In [ ]:
from collections import Counter

Counter(adata.obs["Clone"].to_list())

## SubRegional Definitions #Target-1

In [ ]:
plot_adata(adata, "Clone", plot_type="spatial")

In [ ]:
plot_adata(adata,"L0_leiden_assigned_celltype",plot_type="spatial")

In [ ]:
adata_left = filter_adata_by_region(adata, (0,5178),(685,6915))
adata_right = filter_adata_by_region(adata, (5178,11000),(685,6915))

In [ ]:
plot_adata(adata_right, "L0_leiden_assigned_celltype", plot_type="spatial")

In [ ]:
plot_adata(adata_left, "L0_leiden_assigned_celltype", plot_type="spatial")

In [ ]:
import anndata as ad

adatas = {"LEFT": adata_left, "RIGHT": adata_right}

for location, adata in adatas.items():
    adata.obs["location"] = location

adata_merged = ad.concat(list(adatas.values()), join="outer", label="location", keys=list(adatas.keys()))

adata_merged.obs["location_assigned_celltype"] = (
    adata_merged.obs["location"].astype(str) + "_" + adata_merged.obs["assigned_celltype_L0"].astype(str)
)

In [ ]:
counts_dict = {}
for label,adata in adatas.items():
    counts_dict[label] = adata.obs['L0_leiden_assigned_celltype_sq'].value_counts()

df = pd.DataFrame(counts_dict).fillna(0).astype(int)

df = df.sort_index()

ax = df.plot(
    kind="bar",
    figsize=(max(10, df.shape[0]*0.8), 10),  # auto-size for many cell types
    width=0.8
)
plt.xlabel("Cell Type")
plt.ylabel("Cell Count")
plt.title("Cell Type Distributions Across Datasets")
plt.legend(title="Dataset", bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

# Assesment for Tumors for Target-1

In [ ]:
res = de_volcano(adata_merged, "location_assigned_celltype","RIGHT_tumor_cell_markers", "LEFT_tumor_cell_markers",lfc_thresh=1.5,pval_thresh=0.005,top_n=20)
res[(res["pvals_adj"] < 0.005) & (res["logfoldchanges"] > abs(1.5))]

In [ ]:
go_fgsea(adata_merged, "MIDDLE_tumor_cell_markers", "LEFT_tumor_cell_markers", io_key="location_assigned_celltype")

# Assesment for CD4 T Cells for Target-1

In [ ]:
res = de_volcano(adata_merged, "location_assigned_celltype","LEFT_CD4 T Cell", "MIDDLE_CD4 T Cell",lfc_thresh=1.5,pval_thresh=0.05,top_n=20)
sig_res = res[
    (res["pvals_adj"] < 0.05) &
    (res["logfoldchanges"].abs() > 1.5)
].copy()

sig_res

In [ ]:
go_fgsea(adata_merged,"LEFT_CD4 T Cell", "MIDDLE_CD4 T Cell", io_key="location_assigned_celltype", gene_set="GO_Biological_Process_2025")

# Assesment for CD8 T Cells for Target-1

In [ ]:
res = de_volcano(adata_merged, "location_assigned_celltype","LEFT_CD8 T Cell", "MIDDLE_CD8 T Cell",lfc_thresh=1.5,pval_thresh=0.05,top_n=20)
sig_res = res[
    (res["pvals_adj"] < 0.05) &
    (res["logfoldchanges"].abs() > 1.5)
].copy()

sig_res

In [ ]:
go_fgsea(adata_merged,"LEFT_CD8 T Cell", "MIDDLE_CD8 T Cell", io_key="location_assigned_celltype", gene_set="GO_Biological_Process_2025")

# Assesment for Macrophages for Target-1

In [ ]:
res = de_volcano(adata_merged, "location_assigned_celltype","LEFT_Macrophage_M2", "MIDDLE_Macrophage_M2",lfc_thresh=1.5,pval_thresh=0.05,top_n=20)
sig_res = res[
    (res["pvals_adj"] < 0.05) &
    (res["logfoldchanges"].abs() > 1.5)
].copy()

sig_res

In [ ]:
go_fgsea(adata_merged,"LEFT_Macrophage_M2", "MIDDLE_Macrophage_M2", io_key="location_assigned_celltype", gene_set="GO_Biological_Process_2025")

# Assesment for Endothelial for Target-1

In [ ]:
res = de_volcano(adata_merged, "location_assigned_celltype","LEFT_prostate_gland_microvascular_endothelial_cell", "MIDDLE_prostate_gland_microvascular_endothelial_cell",lfc_thresh=1.5,pval_thresh=0.05,top_n=20)
sig_res = res[
    (res["pvals_adj"] < 0.05) &
    (res["logfoldchanges"].abs() > 1.5)
].copy()

sig_res

In [ ]:
go_fgsea(adata_merged,"LEFT_prostate_gland_microvascular_endothelial_cell", "MIDDLE_prostate_gland_microvascular_endothelial_cell", io_key="location_assigned_celltype", gene_set="KEGG_2016")

# Assesment for Fibroblasts #Target-1

In [ ]:
res = de_volcano(adata_merged, "location_assigned_celltype","LEFT_fibroblast_of_connective_tissue_of_nonglandular_part_of_prostate", "MIDDLE_fibroblast_of_connective_tissue_of_nonglandular_part_of_prostate",lfc_thresh=1.5,pval_thresh=0.05,top_n=20)
sig_res = res[
    (res["pvals_adj"] < 0.05) &
    (res["logfoldchanges"].abs() > 1.5)
].copy()

sig_res

In [ ]:
go_fgsea(adata_merged,"LEFT_fibroblast_of_connective_tissue_of_glandular_part_of_prostate", "MIDDLE_fibroblast_of_connective_tissue_of_glandular_part_of_prostate", io_key="location_assigned_celltype", gene_set="KEGG_2016")

# Receptor Ligand Interaction for Regions

In [ ]:
import multiprocessing

if __name__ == "__main__":
        multiprocessing.freeze_support()

        res_left = run_ligrec(adata_left)

plot_ligrec(res_left, 'CD8 T Cell', "tumor_cell_markers")

In [ ]:
import multiprocessing

if __name__ == "__main__":
        multiprocessing.freeze_support()

        res_left = run_ligrec(adata_middle)

plot_ligrec(res_left, 'CD8 T Cell', "tumor_cell_markers")

## SubRegional Definitions #Target-2

In [ ]:
adata.obs.L0_leiden_assigned_celltype_sq.unique().to_list()